# Agglomerative Clustering — Agrupamiento de series de cultivos (EVA)

**Inteligencia Artificial I — Grupo C2, 2026-2**

Modelo no supervisado: **Agglomerative Clustering** (jerárquico). Empieza con cada serie (municipio + cultivo) como un cluster y va **uniendo los dos más parecidos** hasta que queda uno solo. El resultado es un árbol (dendrograma) que se "corta" en el número de clusters que se quiera.

| Fase | Qué hace |
|---|---|
| 1 | Limpieza de datos (la misma de la fase supervisada) |
| 2 | Pipeline de series: 4 features por serie, filtro y escalado |
| 3 | Elegir el **número de clusters** con el dendrograma |
| 4 | Modelo final: métricas y perfil de cada cluster |
| 5 | Gráficas para interpretar y comparar con los otros modelos |
| 6 | Guardar los CSV |
| 7 | Puente con la fase supervisada: WAPE del SVR por cluster |

**Nota:** este modelo calcula las distancias entre todas las series, así que usa cerca de 1,2 GB de memoria y tarda unos segundos más que los otros.

In [ ]:
# =========================================================
# Librerias que vamos a usar
# =========================================================
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Estilo de las graficas
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["font.size"] = 11

VERDE = "#2a9d8f"
NARANJA = "#e76f51"
AZUL = "#264653"
GRIS = "#bbbbbb"

# Para ver bien las tablas
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

SEMILLA = 42
print("Librerias cargadas correctamente")

## Fase 1 — Limpieza de datos

Es la misma limpieza de la fase supervisada. Deja la tabla `datos` con una fila por municipio + cultivo + estado físico + año.

In [ ]:
# =====================================================================
# LIMPIEZA DEFINITIVA DE LOS ARCHIVOS EVA (2007-2018 y 2019-2025)
# Resultado: la tabla "datos" con UNA fila por municipio + cultivo +
# estado fisico + anio, lista para entrenar los modelos.
# =====================================================================

import pandas as pd
import numpy as np

# =========================================================
# PASO 1: Cargar los dos archivos desde Google Drive
# =========================================================
print("PASO 1: Cargar los dos archivos desde Google Drive")
# Los archivos deben estar guardados en:  MiUnidad / EVA /
from google.colab import drive
drive.mount("/content/drive")

RUTA = "/content/drive/MyDrive/EVA/"

# Leemos todo como texto: los numeros vienen en formato colombiano
# ("1.247.206,27") y los codigos DANE tienen ceros a la izquierda ("05001").
eva_viejo = pd.read_csv(RUTA + "EVA_2007_2018.csv", dtype=str)
eva_nuevo = pd.read_csv(RUTA + "EVA_2019_2025.csv", dtype=str)

print("EVA 2007-2018:", eva_viejo.shape[0], "filas y", eva_viejo.shape[1], "columnas")
print("EVA 2019-2025:", eva_nuevo.shape[0], "filas y", eva_nuevo.shape[1], "columnas")
print("PASO 1: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 2: Dejar solo las columnas necesarias, con el mismo nombre
# =========================================================
print("PASO 2: Dejar solo las columnas necesarias, con el mismo nombre")

# El archivo viejo trae saltos de linea dentro de los nombres
# ("Área Sembrada\n(ha)"): los cambiamos por un espacio.
eva_viejo.columns = [" ".join(nombre.split()) for nombre in eva_viejo.columns]
eva_nuevo.columns = [" ".join(nombre.split()) for nombre in eva_nuevo.columns]

# Solo columnas que existen en LOS DOS archivos y que se usan despues.
# (El rendimiento no se carga: se recalcula al final con la formula.)
nombres_nuevo = {
    "Código Dane departamento":  "cod_dep",
    "Departamento":              "departamento",
    "Código Dane municipio":     "cod_mun",
    "Municipio":                 "municipio",
    "Grupo cultivo":             "grupo",
    "Cultivo":                   "cultivo",
    "Desagregación cultivo":     "desagregacion",
    "Año":                       "anio",
    "Periodo":                   "periodo",
    "Área sembrada":             "area_sembrada",
    "Área cosechada":            "area_cosechada",
    "Producción":                "produccion",
    "Ciclo del cultivo":         "ciclo",
    "Estado físico del cultivo": "estado_fisico",
}
nombres_viejo = {
    "CÓD. DEP.":                                     "cod_dep",
    "DEPARTAMENTO":                                  "departamento",
    "CÓD. MUN.":                                     "cod_mun",
    "MUNICIPIO":                                     "municipio",
    "GRUPO DE CULTIVO":                              "grupo",
    "CULTIVO":                                       "cultivo",
    "DESAGREGACIÓN REGIONAL Y/O SISTEMA PRODUCTIVO": "desagregacion",
    "AÑO":                                           "anio",
    "PERIODO":                                       "periodo",
    "Área Sembrada (ha)":                            "area_sembrada",
    "Área Cosechada (ha)":                           "area_cosechada",
    "Producción (t)":                                "produccion",
    "CICLO DE CULTIVO":                              "ciclo",
    "ESTADO FISICO PRODUCCION":                      "estado_fisico",
}
COLUMNAS = list(nombres_nuevo.values())
eva_nuevo = eva_nuevo.rename(columns=nombres_nuevo)[COLUMNAS].copy()
eva_viejo = eva_viejo.rename(columns=nombres_viejo)[COLUMNAS].copy()

print("Columnas que se conservan:", COLUMNAS)
print("PASO 2: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 3: Convertir los numeros y los codigos DANE
# =========================================================
print("PASO 3: Convertir los numeros y los codigos DANE")

def convertir_a_numero(columna):
    # "1.247.206,27" -> 1247206.27  (el punto separa miles, la coma es decimal)
    texto = columna.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(texto, errors="coerce")

for df in (eva_viejo, eva_nuevo):
    for columna in ["anio", "area_sembrada", "area_cosechada", "produccion"]:
        df[columna] = convertir_a_numero(df[columna])
    df["anio"] = df["anio"].astype(int)
    # Codigos: "15.114" -> "15114"  y  "5.001" -> "05001"
    df["cod_dep"] = df["cod_dep"].str.replace(".", "", regex=False).str.zfill(2)
    df["cod_mun"] = df["cod_mun"].str.replace(".", "", regex=False).str.zfill(5)

# Controles: ningun numero vacio, ningun negativo, codigos coherentes
for nombre, df in [("2007-2018", eva_viejo), ("2019-2025", eva_nuevo)]:
    numeros = df[["area_sembrada", "area_cosechada", "produccion"]]
    assert numeros.notna().all().all(), "Hay numeros que no se pudieron convertir en " + nombre
    assert (numeros >= 0).all().all(), "Hay numeros negativos en " + nombre
    assert (df["cod_mun"].str[:2] == df["cod_dep"]).all(), "Codigos inconsistentes en " + nombre

print("PASO 3: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 4: Poner el texto en mayusculas y sin tildes
# =========================================================
print("PASO 4: Poner el texto en mayusculas y sin tildes")

def limpiar_texto(columna):
    texto = columna.str.upper().str.strip()
    texto = texto.str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("utf-8")
    return texto.str.replace(r"\s+", " ", regex=True)   # espacios dobles -> uno

for df in (eva_viejo, eva_nuevo):
    for columna in ["departamento", "municipio", "grupo", "cultivo",
                    "desagregacion", "ciclo", "estado_fisico"]:
        df[columna] = limpiar_texto(df[columna])

print("PASO 4: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 5: Estandarizar los nombres de cultivo y de estado fisico
# =========================================================
# Regla: solo se unifica lo que es el MISMO cultivo. Lo que es distinto
# se deja separado, y si queda solo en un archivo se elimina en el PASO 6.
print("PASO 5: Estandarizar los nombres de cultivo y de estado fisico")

# 5.1 El archivo nuevo junta las tres canas en "CANA", pero la desagregacion
#     las separa. Como son productos distintos, las separamos igual que el viejo.
es_cana = eva_nuevo["cultivo"] == "CANA"
eva_nuevo.loc[es_cana, "cultivo"] = eva_nuevo.loc[es_cana, "desagregacion"]
eva_nuevo["cultivo"] = eva_nuevo["cultivo"].replace({"CANA DE AZUCAR": "CANA AZUCARERA",
                                                     "ESTEVIA": "STEVIA"})

# 5.2 Nombres del archivo viejo que se pasan al nombre del archivo nuevo.
cultivos_equivalentes = {
    # a) El mismo cultivo, solo cambio el nombre
    "DURAZNO":     "DURAZNO O ALBARICOQUE",
    "AGRAZ":       "AGRAZ - MORTINO",
    "PEPA DE PAN": "ARBOL DE PAN O PEPA DEL PAN",
    "PAPAYUELA":   "PAPAYUELA -BABACO",
    "CURCUMA":     "CURCUMA O AZAFRAN",
    "RUBA":        "ULLUCO",
    # b) El archivo nuevo definio una categoria que junta exactamente
    #    los cultivos que el viejo traia separados
    "GULUPA":      "GULUPA O CHOLUPA",
    "CHOLUPA":     "GULUPA O CHOLUPA",
    "CALABACIN":   "CALABACIN, CALABAZA",
    "CALABAZA":    "CALABACIN, CALABAZA",
    "MALANGA":     "MALANGA, ACHIN, YOTA, PAPA CHINA, BORE",
    "BORE":        "MALANGA, ACHIN, YOTA, PAPA CHINA, BORE",
}
eva_viejo["cultivo"] = eva_viejo["cultivo"].replace(cultivos_equivalentes)

# 5.3 Estado fisico: se unifican solo los nombres que significan lo mismo.
#     Lo demas se deja como viene (ej. PANELA y CANA O VERDE son distintos).
estados_equivalentes = {
    "FRUTO FRESCO":       "EN FRESCO",
    "HORTALIZA FRESCA":   "EN FRESCO",
    "TUBERCULO FRESCO":   "EN FRESCO",
    "HOJA FRESCA":        "EN FRESCO",
    "LEGUMINOSA FRESCA":  "EN FRESCO",
    "PADDY VERDE":        "PADDY O CASCARA VERDE",
    "SEMILLA":            "ALGODON SEMILLA",
    "CANA PARA MOLIENDA": "CANA O VERDE",
}
eva_viejo["estado_fisico"] = eva_viejo["estado_fisico"].replace(estados_equivalentes)

print("PASO 5: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 6: Mismo grupo y ciclo para cada cultivo, y solo cultivos de ambos archivos
# =========================================================
print("PASO 6: Mismo grupo y ciclo para cada cultivo, y solo cultivos de ambos archivos")

# 6.1 Nos quedamos solo con los cultivos que aparecen en LOS DOS archivos
cultivos_comunes = set(eva_viejo["cultivo"]) & set(eva_nuevo["cultivo"])
eliminados = (pd.concat([eva_viejo, eva_nuevo])
                .loc[lambda df: ~df["cultivo"].isin(cultivos_comunes)]
                .groupby("cultivo")["area_cosechada"].sum()
                .sort_values(ascending=False))
print("Cultivos que estan en un solo archivo (se eliminan):", len(eliminados))
print("Los 10 con mas area cosechada:")
print(eliminados.head(10).round(0))

eva_viejo = eva_viejo[eva_viejo["cultivo"].isin(cultivos_comunes)].copy()
eva_nuevo = eva_nuevo[eva_nuevo["cultivo"].isin(cultivos_comunes)].copy()

# 6.2 Catalogo: grupo y ciclo de cada cultivo segun el archivo nuevo
#     (ahi cada cultivo tiene un solo grupo y un solo ciclo)
catalogo = eva_nuevo.groupby("cultivo")[["grupo", "ciclo"]].first()
assert (eva_nuevo.groupby("cultivo")["grupo"].nunique() == 1).all()
assert (eva_nuevo.groupby("cultivo")["ciclo"].nunique() == 1).all()

# 6.3 El archivo viejo toma el grupo y el ciclo del catalogo
eva_viejo["grupo"] = eva_viejo["cultivo"].map(catalogo["grupo"])
eva_viejo["ciclo"] = eva_viejo["cultivo"].map(catalogo["ciclo"])

print("Cultivos que quedan:", len(cultivos_comunes))
print("Grupos:", sorted(catalogo["grupo"].unique()))
print("PASO 6: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 7: Unir los dos archivos y quitar los anios 2006 y 2018
# =========================================================
print("PASO 7: Unir los dos archivos y quitar los anios 2006 y 2018")

eva_viejo["fuente"] = "2007-2018"
eva_nuevo["fuente"] = "2019-2025"
datos = pd.concat([eva_viejo, eva_nuevo], ignore_index=True)

# Nombres oficiales a partir del codigo DANE (se usa el del archivo nuevo).
# Asi un municipio no aparece con dos nombres ("DON MATIAS" / "DONMATIAS").
for codigo, nombre in [("cod_dep", "departamento"), ("cod_mun", "municipio")]:
    nombre_oficial = datos.sort_values("fuente").groupby(codigo)[nombre].last()
    datos[nombre] = datos[codigo].map(nombre_oficial)

# 2006: solo trae el semestre B.  2018: no trae el semestre B.
# Los dos anios estan incompletos y sesgarian los totales anuales.
print("Filas del 2006:", (datos["anio"] == 2006).sum())
print("Filas del 2018:", (datos["anio"] == 2018).sum())
datos = datos[~datos["anio"].isin([2006, 2018])]

print("Filas despues de unir:", len(datos))
print("PASO 7: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 8: Pasar de semestres a anios (suma simple)
# =========================================================
# Se suman los semestres A y B y las variedades (desagregaciones) de cada
# cultivo. La produccion solo se suma entre filas con el MISMO estado fisico,
# por eso el estado fisico es parte de la clave (ej. maiz en grano seco y
# maiz en fresco quedan en filas separadas).
print("PASO 8: Pasar de semestres a anios (suma simple)")

CLAVE = ["cod_mun", "cultivo", "estado_fisico", "anio"]
datos = (datos.groupby(CLAVE, as_index=False)
              .agg(fuente=("fuente", "first"),
                   cod_dep=("cod_dep", "first"),
                   departamento=("departamento", "first"),
                   municipio=("municipio", "first"),
                   grupo=("grupo", "first"),
                   ciclo=("ciclo", "first"),
                   area_sembrada=("area_sembrada", "sum"),
                   area_cosechada=("area_cosechada", "sum"),
                   produccion=("produccion", "sum")))

# Rendimiento recalculado con la formula (vacio si no hubo cosecha)
datos["rendimiento"] = np.where(datos["area_cosechada"] > 0,
                                datos["produccion"] / datos["area_cosechada"], np.nan)

print("Filas anuales:", len(datos))
print("PASO 8: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 9: Regla de arrastre (sin tolerancia) y consistencia
# =========================================================
# En los cultivos TRANSITORIOS, lo sembrado y no cosechado un anio puede
# cosecharse al anio siguiente (solo se arrastra UN anio):
#     area_pendiente_anterior = sembrada(anio-1) - cosechada(anio-1)   (si es > 0)
#     area_disponible         = sembrada(anio) + area_pendiente_anterior
# En los PERMANENTES no hay arrastre (el area sembrada ya es todo lo plantado).
# Se elimina toda fila que coseche mas que el area disponible.
print("PASO 9: Regla de arrastre (sin tolerancia) y consistencia")

datos = datos.sort_values(CLAVE).reset_index(drop=True)
serie = datos.groupby(["cod_mun", "cultivo", "estado_fisico"])
sembrada_ant = serie["area_sembrada"].shift(1)
cosechada_ant = serie["area_cosechada"].shift(1)
anio_ant = serie["anio"].shift(1)

# Solo hay arrastre si existe el anio inmediatamente anterior y es transitorio
hay_arrastre = (datos["anio"] - anio_ant == 1) & (datos["ciclo"] == "TRANSITORIO")
datos["area_pendiente_anterior"] = np.where(hay_arrastre,
                                            (sembrada_ant - cosechada_ant).clip(lower=0), 0.0)
area_disponible = datos["area_sembrada"] + datos["area_pendiente_anterior"]

regla_arrastre = datos["area_cosechada"] > area_disponible        # cosecho mas de lo disponible
regla_produccion = (datos["produccion"] > 0) & (datos["area_cosechada"] == 0)   # produccion sin cosecha
sin_area = area_disponible == 0                                    # nada sembrado ni pendiente

print("Cosecharon mas que el area disponible:", regla_arrastre.sum())
print("Produccion sin area cosechada        :", regla_produccion.sum())
print("Sin area sembrada ni pendiente       :", sin_area.sum())
datos = datos[~(regla_arrastre | regla_produccion | sin_area)].reset_index(drop=True)

print("Filas que quedan:", len(datos))
print("PASO 9: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 10: Agregar la altura de cada municipio (msnm)
# =========================================================
# Fuente: IGAC, publicada en el Panel Municipal del CEDE (Universidad de los
# Andes), archivo "PANEL_CARACTERISTICAS_GENERALES_2021.csv" guardado en la
# misma carpeta de Drive. Columnas usadas: codmpio (codigo DANE) y altura (msnm).
print("PASO 10: Agregar la altura de cada municipio (msnm)")

ARCHIVO_ALTURA = RUTA + "PANEL_CARACTERISTICAS_GENERALES_2021.csv"
altura = pd.read_csv(ARCHIVO_ALTURA, usecols=["codmpio", "altura"], encoding="latin-1")

# El panel trae una fila por anio, pero la altura de un municipio no cambia:
# nos quedamos con una sola fila por municipio.
altura = altura.dropna(subset=["altura"]).drop_duplicates(subset="codmpio")
altura["cod_mun"] = altura["codmpio"].astype(int).astype(str).str.zfill(5)   # 5001 -> "05001"
altura = altura.rename(columns={"altura": "altura_msnm"})[["cod_mun", "altura_msnm"]]

# Municipios que no estan en el panel (creados despues): altura aproximada a mano
altura_manual = pd.DataFrame({
    "cod_mun":     ["27493"],     # NUEVO BELEN DE BAJIRA (Choco)
    "altura_msnm": [105],
})
altura = pd.concat([altura, altura_manual], ignore_index=True)

datos = datos.merge(altura, on="cod_mun", how="left")

print("PASO 10: Ejecutado correctamente")
print("-------------------------------")

# =========================================================
# PASO 11: Revision final y orden de columnas
# =========================================================
print("PASO 11: Revision final y orden de columnas")

# La clave debe ser unica: una fila por municipio + cultivo + estado + anio
assert not datos.duplicated(CLAVE).any(), "Hay filas repetidas en la clave"

COLUMNAS_FINALES = ["anio", "fuente", "cod_dep", "departamento", "cod_mun", "municipio",
                    "altura_msnm", "grupo", "cultivo", "ciclo", "estado_fisico",
                    "area_sembrada", "area_pendiente_anterior", "area_cosechada",
                    "produccion", "rendimiento"]
datos = datos[COLUMNAS_FINALES]

print("Filas:", len(datos), "| Columnas:", datos.shape[1])
print("Anios:", sorted(datos["anio"].unique()))
print("Departamentos:", datos["cod_dep"].nunique(), "| Municipios:", datos["cod_mun"].nunique(),
      "| Cultivos:", datos["cultivo"].nunique(), "| Grupos:", datos["grupo"].nunique())
print()
print("Datos faltantes por columna:")
print(datos.isna().sum()[datos.isna().sum() > 0])
print("(el rendimiento queda vacio solo cuando el area cosechada es 0)")
print()
print("Filas por grupo y archivo:")
print(pd.crosstab(datos["grupo"], datos["fuente"]))
print()
# Cultivos cuyo estado fisico cambia entre archivos (su produccion no se compara)
estados = datos.groupby(["cultivo", "fuente"])["estado_fisico"].apply(lambda s: " / ".join(sorted(s.unique())))
estados = estados.unstack()
cambian = estados[estados["2007-2018"] != estados["2019-2025"]]
print("Cultivos cuyo estado fisico cambia entre archivos:", len(cambian))
print(cambian)
print("PASO 11: Ejecutado correctamente")
print("-------------------------------")

# Para guardar (al leerlo de nuevo, usar dtype={"cod_dep": str, "cod_mun": str}):
# datos.to_csv(RUTA + "eva_limpio.csv", index=False, encoding="utf-8-sig")

datos.head()

## Fase 2 — Pipeline de series

Aquí **no se agrupan filas, se agrupan series**. Una **serie** es un **municipio + cultivo** a lo largo de los años (sin `estado_fisico`, porque cambia entre los dos archivos EVA).

| Paso | Qué hace |
|---|---|
| S1 | Suma las áreas por municipio + cultivo + año y deja solo **2007–2022** (no se usan los años de prueba) |
| S2 | Resume cada serie en **4 features** y deja las series con **3 años o más** de registro |
| S3 | Escala las 4 features con `StandardScaler` (los 3 algoritmos miden distancias) |

| Feature | Cómo se calcula | Qué dice de la serie |
|---|---|---|
| `log_area_media` | log(1 + promedio del área disponible) | Qué tan grande es |
| `tasa_media` | total cosechado / total disponible | Cuánto cosecha normalmente |
| `desv_tasa` | desviación estándar de la tasa de cada año | Qué tan estable es su cosecha |
| `cv_area` | desviación estándar del área sembrada / su promedio | Qué tanto cambia el área que se siembra |

`ciclo`, `grupo` y `departamento` **no entran al agrupamiento**: se guardan para interpretar los clusters después.

In [ ]:
# =========================================================
# PASO S1: Armar las series (municipio + cultivo) por anio
# =========================================================
print("PASO S1: Armar las series por anio")
ANIO_FIN = 2022          # solo 2007-2022: los anios de prueba (2023-2025) no se usan
MIN_ANIOS = 3            # una serie debe tener 3 anios o mas de registro

SERIE = ["cod_mun", "cultivo"]
AREAS = ["area_sembrada", "area_pendiente_anterior", "area_cosechada"]

# Una fila por municipio + cultivo + anio (se suman los estados fisicos)
anual = datos[datos["anio"] <= ANIO_FIN].groupby(SERIE + ["anio"], as_index=False)[AREAS].sum()

# Area que habia para cosechar y tasa de cosecha de cada anio
anual["area_disponible"] = anual["area_sembrada"] + anual["area_pendiente_anterior"]
anual["tasa"] = anual["area_cosechada"] / anual["area_disponible"]

print("Filas (municipio + cultivo + anio):", len(anual))
print("Series (municipio + cultivo):", anual.groupby(SERIE).ngroups)
print("PASO S1: Ejecutado correctamente")
print("-------------------------------")

In [ ]:
# =========================================================
# PASO S2: Resumir cada serie en 4 features y filtrar
# =========================================================
print("PASO S2: Resumir cada serie en 4 features")
FEATURES = ["log_area_media", "tasa_media", "desv_tasa", "cv_area"]

g = anual.groupby(SERIE)
series = pd.DataFrame({
    "anios": g["anio"].count(),                                     # anios con registro
    "area_media": g["area_disponible"].mean(),                      # hectareas promedio
    "tasa_media": g["area_cosechada"].sum() / g["area_disponible"].sum(),
    "desv_tasa": g["tasa"].std(),
    "cv_area": g["area_sembrada"].std() / g["area_sembrada"].mean(),
}).reset_index()
series["log_area_media"] = np.log1p(series["area_media"])

# Filtro: solo series con 3 anios o mas (sin limite de hectareas)
total = len(series)
series = series[series["anios"] >= MIN_ANIOS].reset_index(drop=True)

# Columnas para interpretar los clusters (NO entran al agrupamiento)
info = datos.drop_duplicates(SERIE)[SERIE + ["municipio", "departamento", "grupo", "ciclo"]]
series = series.merge(info, on=SERIE, how="left")

print("Series totales:", total, "| Series con", MIN_ANIOS, "anios o mas:", len(series))
print("Datos faltantes en las features:", int(series[FEATURES].isna().sum().sum()))
print()
print(series[FEATURES].describe().round(3).to_string())
print("PASO S2: Ejecutado correctamente")
print("-------------------------------")

In [ ]:
# =========================================================
# PASO S3: Escalar las features
# =========================================================
print("PASO S3: Escalar las features")
# Los 3 algoritmos miden distancias: todas las features deben pesar igual
X = StandardScaler().fit_transform(series[FEATURES])

print("Matriz para agrupar:", X.shape[0], "series x", X.shape[1], "features")
print("PASO S3: Ejecutado correctamente")
print("-------------------------------")

## Fase 3 — Elegir el número de clusters

Se usa el enlace **Ward**: en cada paso une los dos clusters que menos aumentan la dispersión interna (es el más parecido a K-Means).

- **Dendrograma:** muestra las últimas 30 uniones del árbol. La **altura** de cada unión es la distancia entre los dos clusters que se unen. Se corta donde hay un **salto grande de altura**: unir más allá de ese punto sería juntar clusters muy distintos. El número de líneas verticales que cruza el corte es el número de clusters.
- **Silhouette según el número de clusters:** para confirmar el corte.

Después de ver las gráficas, escribir el número de clusters elegido en **H2**.

In [ ]:
# =========================================================
# PASO H1: Dendrograma y silhouette
# =========================================================
print("PASO H1: Dendrograma y silhouette")
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Arbol completo con el enlace Ward (tarda unos segundos)
arbol = linkage(X, method="ward")

plt.figure(figsize=(13, 5))
dendrogram(arbol, truncate_mode="lastp", p=30, no_labels=True, color_threshold=0, above_threshold_color=AZUL)
plt.xlabel("Ultimas 30 uniones del arbol (cada linea de abajo es un grupo de series)")
plt.ylabel("Altura de la union (distancia Ward)")
plt.title("Dendrograma")
plt.tight_layout()
plt.show()

# Silhouette al cortar el arbol en 2, 3, ..., 10 clusters
pruebas = []
for k in range(2, 11):
    e = fcluster(arbol, k, criterion="maxclust")
    pruebas.append({"clusters": k, "altura_del_corte": arbol[-(k - 1), 2],
                    "silhouette": silhouette_score(X, e), "davies_bouldin": davies_bouldin_score(X, e)})
pruebas = pd.DataFrame(pruebas)

plt.figure(figsize=(8, 4.5))
plt.plot(pruebas["clusters"], pruebas["silhouette"], marker="o", color=NARANJA)
plt.xlabel("Numero de clusters")
plt.ylabel("Silhouette (mas alto = mejor)")
plt.title("Silhouette segun el numero de clusters")
plt.tight_layout()
plt.show()

print(pruebas.round(3).to_string(index=False))
print("PASO H1: Ejecutado correctamente")
print("-------------------------------")

In [ ]:
# =========================================================
# PASO H2: Hiperparametros elegidos  (EDITAR AQUI)
# =========================================================
print("PASO H2: Hiperparametros elegidos")

K = 4               # numero de clusters elegido con el dendrograma de H1
ENLACE = "ward"     # criterio para unir clusters

print("n_clusters =", K, "| linkage =", ENLACE)
print("PASO H2: Ejecutado correctamente")
print("-------------------------------")

## Modelo final

Se entrena con los hiperparámetros de **H2**. Si se cambian, volver a ejecutar desde H2 hacia abajo.

In [ ]:
# =========================================================
# PASO M1: Entrenar el modelo final
# =========================================================
print("PASO M1: Entrenar Agglomerative Clustering")
NOMBRE_MODELO = "Aglomerativo"
HIPERPARAMETROS = "n_clusters=" + str(K) + ", linkage=" + ENLACE

inicio = time.time()
modelo = AgglomerativeClustering(n_clusters=K, linkage=ENLACE).fit(X)
etiquetas = modelo.labels_
segundos = round(time.time() - inicio, 1)

print("Hiperparametros:", HIPERPARAMETROS, "| Tiempo:", segundos, "s")
print("PASO M1: Ejecutado correctamente")
print("-------------------------------")

## Fase 4 — Resultados del modelo

**Métricas** (se calculan sin las series marcadas como ruido):

| Métrica | Qué mide | Mejor si |
|---|---|---|
| Silhouette | Qué tan cerca está cada serie de su cluster frente al cluster vecino (−1 a 1) | Más alto |
| Davies-Bouldin | Qué tan parecidos son los clusters entre sí | Más bajo |
| Calinski-Harabasz | Separación entre clusters frente a dispersión dentro de ellos | Más alto |

Los clusters se renumeran por tamaño: **el cluster 0 es el más grande**. El ruido (solo DBSCAN) queda como **−1**.

In [ ]:
# =========================================================
# PASO R1: Metricas del agrupamiento
# =========================================================
print("PASO R1: Metricas del agrupamiento")

# Renumerar: 0 = cluster mas grande. El ruido (-1) se queda como -1
conteo = pd.Series(etiquetas[etiquetas >= 0]).value_counts()
nuevo = {viejo: i for i, viejo in enumerate(conteo.index)}
nuevo[-1] = -1
series["cluster"] = [nuevo[e] for e in etiquetas]

sin_ruido = (series["cluster"] >= 0).values
n_clusters = series.loc[sin_ruido, "cluster"].nunique()
tamanos = series.loc[sin_ruido, "cluster"].value_counts()

metricas_modelo = {
    "modelo": NOMBRE_MODELO,
    "hiperparametros": HIPERPARAMETROS,
    "series": len(series),
    "n_clusters": n_clusters,
    "ruido_%": (~sin_ruido).mean() * 100,
    "silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan,
    "cluster_mas_grande": tamanos.max(),
    "cluster_mas_pequeno": tamanos.min(),
    "segundos": segundos,
}
if n_clusters >= 2:          # las metricas necesitan al menos 2 clusters
    metricas_modelo["silhouette"] = silhouette_score(X[sin_ruido], series.loc[sin_ruido, "cluster"])
    metricas_modelo["davies_bouldin"] = davies_bouldin_score(X[sin_ruido], series.loc[sin_ruido, "cluster"])
    metricas_modelo["calinski_harabasz"] = calinski_harabasz_score(X[sin_ruido], series.loc[sin_ruido, "cluster"])

for nombre, valor in metricas_modelo.items():
    print(nombre.ljust(20), ":", round(valor, 3) if isinstance(valor, float) else valor)
print("PASO R1: Ejecutado correctamente")
print("-------------------------------")

In [ ]:
# =========================================================
# PASO R2: Perfil de cada cluster (en unidades reales)
# =========================================================
print("PASO R2: Perfil de cada cluster")

def nombre_cluster(c):
    return "Ruido" if c == -1 else "Cluster " + str(c)

g = series.groupby("cluster")
perfiles = pd.DataFrame({
    "series": g.size(),
    "% de las series": g.size() / len(series) * 100,
    "% del area": g["area_media"].sum() / series["area_media"].sum() * 100,
    "area tipica (ha)": np.expm1(g["log_area_media"].mean()),
    "tasa de cosecha (%)": g["tasa_media"].mean() * 100,
    "desv. de la tasa (puntos)": g["desv_tasa"].mean() * 100,
    "cv del area": g["cv_area"].mean(),
    "% permanentes": g["ciclo"].apply(lambda s: (s == "PERMANENTE").mean() * 100),
})
perfiles.index = [nombre_cluster(c) for c in perfiles.index]
print(perfiles.round(1).to_string())

print()
print("Los 3 cultivos con mas series en cada cluster:")
for c, tabla in series.groupby("cluster"):
    print(" ", nombre_cluster(c).ljust(10), "->", ", ".join(tabla["cultivo"].value_counts().head(3).index))
print("PASO R2: Ejecutado correctamente")
print("-------------------------------")

## Fase 5 — Gráficas para interpretar y comparar

Las mismas 3 gráficas en los 3 notebooks, para comparar los modelos uno al lado del otro:

1. **Mapa de las series:** tamaño contra tasa de cosecha, cada punto es una serie y el color es su cluster.
2. **Perfil de los clusters:** rojo = por encima del promedio de todas las series, azul = por debajo. El número es el valor real.
3. **Permanentes y transitorios por cluster:** el `ciclo` no entró al agrupamiento; aquí se ve si los clusters lo separan.

In [ ]:
# =========================================================
# PASO G1: Mapa de las series (tamanio vs tasa de cosecha)
# =========================================================
print("PASO G1: Mapa de las series")
clusters = sorted(c for c in series["cluster"].unique() if c >= 0)
paleta = sns.color_palette("tab10", max(len(clusters), 1))
COLOR_CLUSTER = {c: paleta[i % 10] for i, c in enumerate(clusters)}
COLOR_CLUSTER[-1] = GRIS

plt.figure(figsize=(11, 6))
for c in [-1] + clusters:
    t = series[series["cluster"] == c]
    if len(t) > 0:
        plt.scatter(t["log_area_media"], t["tasa_media"] * 100, s=8, alpha=0.45,
                    color=COLOR_CLUSTER[c], label=nombre_cluster(c) + " (" + str(len(t)) + ")")
hectareas = [1, 10, 100, 1000, 10000]
plt.xticks(np.log1p(hectareas), [str(h) for h in hectareas])
plt.xlabel("Area promedio de la serie (hectareas, escala log)")
plt.ylabel("Tasa de cosecha media (%)")
plt.title(NOMBRE_MODELO + ": mapa de las series por cluster")
leyenda = plt.legend(frameon=False, markerscale=3, loc="center left", bbox_to_anchor=(1, 0.5))
for marca in leyenda.legend_handles:
    marca.set_alpha(1)
plt.tight_layout()
plt.show()
print("PASO G1: Ejecutado correctamente")
print("-------------------------------")

In [ ]:
# =========================================================
# PASO G2: Perfil de los clusters
# =========================================================
print("PASO G2: Perfil de los clusters")
TITULOS = ["Tamanio", "Tasa de\ncosecha", "Variacion\nde la tasa", "Variacion\ndel area"]

# Color: promedio de cada feature ESCALADA (0 = promedio de todas las series)
escaladas = pd.DataFrame(X, columns=FEATURES).groupby(series["cluster"].values).mean()
# Texto: el valor real de cada feature
medias = series.groupby("cluster")[FEATURES].mean()
texto = pd.DataFrame({
    "Tamanio": np.expm1(medias["log_area_media"]).round(0).astype(int).astype(str) + " ha",
    "Tasa de cosecha": (medias["tasa_media"] * 100).round(0).astype(int).astype(str) + " %",
    "Variacion de la tasa": (medias["desv_tasa"] * 100).round(0).astype(int).astype(str) + " pts",
    "Variacion del area": medias["cv_area"].round(2).astype(str),
})
filas = [nombre_cluster(c) + "\n(" + str(n) + " series)" for c, n in series.groupby("cluster").size().items()]

plt.figure(figsize=(9, 1.2 + 0.9 * len(filas)))
sns.heatmap(escaladas.values, annot=texto.values, fmt="", cmap="RdBu_r", center=0, vmin=-2, vmax=2,
            xticklabels=TITULOS, yticklabels=filas, linewidths=1, linecolor="white",
            cbar_kws={"label": "Respecto al promedio (desv. estandar)"})
plt.yticks(rotation=0)
plt.xticks(rotation=0)
plt.title(NOMBRE_MODELO + ": perfil de cada cluster")
plt.tight_layout()
plt.show()
print("PASO G2: Ejecutado correctamente")
print("-------------------------------")

In [ ]:
# =========================================================
# PASO G3: Permanentes y transitorios en cada cluster
# =========================================================
print("PASO G3: Permanentes y transitorios en cada cluster")
ciclos = pd.crosstab(series["cluster"], series["ciclo"], normalize="index") * 100
ciclos.index = [nombre_cluster(c) for c in ciclos.index]

ax = ciclos.plot(kind="barh", stacked=True, color=[VERDE, NARANJA], figsize=(10, 1.5 + 0.6 * len(ciclos)))
for barras in ax.containers:
    textos = [str(round(v)) + " %" if v >= 5 else "" for v in barras.datavalues]
    ax.bar_label(barras, labels=textos, label_type="center", color="white", fontsize=10)
general = (series["ciclo"] == "PERMANENTE").mean() * 100
ax.axvline(general, color="black", linestyle="--", linewidth=1.5,
           label="Todas las series: " + str(round(general)) + " % permanentes")
ax.invert_yaxis()
ax.set_xlabel("% de las series del cluster")
ax.set_title(NOMBRE_MODELO + ": permanentes y transitorios por cluster")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

print(ciclos.round(1).to_string())
print("PASO G3: Ejecutado correctamente")
print("-------------------------------")

## Fase 6 — Guardar los resultados

Se guardan en `EVA/Resultados_clustering/`:

| Archivo | Contenido |
|---|---|
| `clusters_<modelo>.csv` | Una fila por serie: `cod_mun`, `cultivo`, sus features y su **cluster**. Sirve para cruzar con `predicciones_SVR.csv` usando `cod_mun` + `cultivo` |
| `metricas_<modelo>.csv` | Una fila con los hiperparámetros y las métricas, para la tabla comparativa de los 3 modelos |
| `perfiles_<modelo>.csv` | El perfil de cada cluster |

In [ ]:
# =========================================================
# PASO E1: Guardar los archivos del modelo
# =========================================================
print("PASO E1: Guardar los archivos del modelo")
RUTA_CLUSTERING = RUTA + "Resultados_clustering/"
os.makedirs(RUTA_CLUSTERING, exist_ok=True)

columnas = ["cod_mun", "cultivo", "municipio", "departamento", "grupo", "ciclo", "anios",
            "area_media"] + FEATURES + ["cluster"]
salida = series[columnas].copy()
salida["modelo"] = NOMBRE_MODELO
salida.to_csv(RUTA_CLUSTERING + "clusters_" + NOMBRE_MODELO + ".csv", index=False, encoding="utf-8-sig")
pd.DataFrame([metricas_modelo]).to_csv(RUTA_CLUSTERING + "metricas_" + NOMBRE_MODELO + ".csv", index=False)
perfiles.to_csv(RUTA_CLUSTERING + "perfiles_" + NOMBRE_MODELO + ".csv", encoding="utf-8-sig")

print("Guardados en", RUTA_CLUSTERING)
print("  clusters_" + NOMBRE_MODELO + ".csv |", len(salida), "series")
print("  metricas_" + NOMBRE_MODELO + ".csv")
print("  perfiles_" + NOMBRE_MODELO + ".csv")
print("PASO E1: Ejecutado correctamente")
print("-------------------------------")

## Fase 7 — Puente con la fase supervisada

A cada fila de `predicciones_SVR.csv` (prueba 2023–2025) se le asigna el cluster de su serie (`cod_mun` + `cultivo`) y se mide el **WAPE del SVR en cada cluster**. Así se ve si el modelo supervisado falla más en algún tipo de serie.

Las filas **"Sin cluster"** son de series con menos de 3 años de registro en 2007–2022.

Esta celda es independiente: solo lee `clusters_<modelo>.csv` y `predicciones_SVR.csv`, así que se puede mover a otro notebook.

In [ ]:
# =========================================================
# PASO B1: WAPE del SVR en cada cluster (prueba 2023-2025)
# =========================================================
print("PASO B1: WAPE del SVR en cada cluster")
RUTA_EVA = "/content/drive/MyDrive/EVA/"

clus = pd.read_csv(RUTA_EVA + "Resultados_clustering/clusters_" + NOMBRE_MODELO + ".csv", dtype={"cod_mun": str})
pred = pd.read_csv(RUTA_EVA + "Resultados_modelos/predicciones_SVR.csv", dtype={"cod_mun": str})

# Asignar a cada prediccion el cluster de su serie (municipio + cultivo)
cruce = pred.merge(clus[["cod_mun", "cultivo", "cluster"]], on=["cod_mun", "cultivo"], how="left")

def wape(real, estimada):
    return np.abs(real - estimada).sum() / real.sum() * 100

filas = {}
for c in sorted(clus["cluster"].unique()) + ["Sin cluster"]:
    t = cruce[cruce["cluster"].isna()] if c == "Sin cluster" else cruce[cruce["cluster"] == c]
    if len(t) == 0:
        continue
    nombre = c if c == "Sin cluster" else ("Ruido" if c == -1 else "Cluster " + str(c))
    filas[nombre] = {"filas": len(t),
                     "% del area": t["area_cosechada"].sum() / cruce["area_cosechada"].sum() * 100,
                     "WAPE SVR": wape(t["area_cosechada"], t["area_cosechada_estimada"]),
                     "WAPE regla": wape(t["area_cosechada"], t["area_sembrada"])}
puente = pd.DataFrame(filas).T
puente["filas"] = puente["filas"].astype(int)
total_svr = wape(cruce["area_cosechada"], cruce["area_cosechada_estimada"])
print(puente.round(2).to_string())
print("WAPE del SVR en toda la prueba:", round(total_svr, 2), "%")

# Grafica: WAPE del SVR y de la regla simple en cada cluster
# (solo los grupos con 100 filas o mas: con muy pocas filas el WAPE no es confiable)
grafica = puente[puente["filas"] >= 100]
x = np.arange(len(grafica))
plt.figure(figsize=(max(9, 1.8 * len(grafica)), 5.5))
b1 = plt.bar(x - 0.2, grafica["WAPE SVR"], 0.4, color="#264653", label="SVR")
b2 = plt.bar(x + 0.2, grafica["WAPE regla"], 0.4, color="#bbbbbb", label="Regla: cosechada = sembrada")
plt.bar_label(b1, fmt="%.1f", fontsize=9)
plt.bar_label(b2, fmt="%.1f", fontsize=9)
plt.axhline(total_svr, color="#e76f51", linestyle="--", label="SVR en toda la prueba")
plt.xticks(x, [n + "\n" + str(round(a, 1)) + " % del area" for n, a in grafica["% del area"].items()])
plt.ylabel("WAPE (%)")
plt.title("WAPE del SVR en prueba (2023-2025) por cluster de " + NOMBRE_MODELO)
plt.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
plt.tight_layout()
plt.show()
print("PASO B1: Ejecutado correctamente")
print("-------------------------------")